# Marcha Blanca: Evaluación y Asignación de Micro-Clusters (v3.0)
**TuBoleta - Data Science & Machine Learning**

Este cuaderno implementa el proceso de **Marcha Blanca (*Shadow Testing*)** para evaluar el modelo jerárquico de micro-clusters (v3.0) sobre nuevos eventos curados provenientes de Secutix.

### Objetivos del Proceso:
1. **Cero impacto en producción:** Evaluar el comportamiento del modelo de forma aislada.
2. **Asignación en 2 Niveles:** Clasificar cada localidad en su **Macro-Arquetipo (Nivel 1)** y su **Micro-Cluster Canónico (Nivel 2, 19 niveles)**.
3. **Auditoría de Calidad:** Medir deriva poblacional (**PSI - Population Stability Index**) y tasa de incertidumbre (**Fronteras**).
4. **Entrega a Modelos Downstream:** Persistir la tabla enriquecida en la capa Gold para consumo directo por modelos de Pricing y Forecasting.

In [ ]:
# 1. Importación Ligera de Librerías y Módulos de Inferencia
import os
import sys
import joblib
import numpy as np
import pandas as pd

# Configurar paths para módulos locales si se ejecuta desde repo
for p in ['..', '.']:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.jerarquia import predecir_microclusters, evaluar_drift_microclusters, MICRO_CLUSTERS_CANONICAL

print(' Módulos de inferencia y jerarquía importados correctamente.')

In [ ]:
# 2. Configuración de Rutas y Credenciales desde archivo .env
try:
    from dotenv import load_dotenv
    env_path = next((p for p in ['../.env', '.env'] if os.path.exists(p)), None)
    if env_path:
        load_dotenv(env_path)
        print(f"Variables cargadas desde archivo: {env_path}")
except ImportError:
    print("python-dotenv no encontrado, leyendo variables de entorno del sistema.")

# Lectura de credenciales seguras exclusivamente desde entorno/.env (sin secretos hardcodeados)
storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME", "aybdwhstorage01")
storage_account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY", "")
container_name = os.getenv("AZURE_CONTAINER_NAME", "tuboleta")

print(f"Cuenta de Storage: {storage_account_name}")
print(f"Contenedor: {container_name}")
print(f"Credencial Azure configurada: {'[OK]' if storage_account_key else '[FALTA_CLAVE]'}")

# Rutas en Azure Blob Storage (WASBS Gold Layer)
input_path_gold = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/GOLD/SECUTIX/Training Data/Clustering de Localidades test/"
output_path_marcha_blanca = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/GOLD/SECUTIX/Predictions/Clustering_Marcha_Blanca_v3/"

# Ruta local de fallback si no se ejecuta en cluster Spark
candidatos_raw = [
    '../data/raw/localidades_eda.parquet',
    'data/raw/localidades_eda.parquet',
    '../data/processed/localidades_clusterizadas.parquet',
    'data/processed/localidades_clusterizadas.parquet'
]
local_fallback_path = next((p for p in candidatos_raw if os.path.exists(p)), 'data/raw/localidades_eda.parquet')
print(f"Ruta local de contingencia: {local_fallback_path}")

# Búsqueda automática y robusta del modelo jerárquico v3.0
candidatos_modelo = [
    '/dbfs/FileStore/models/modelo_jerarquia_v3.joblib',
    '../data/processed/modelo_jerarquia_v3.joblib',
    'data/processed/modelo_jerarquia_v3.joblib',
    os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed', 'modelo_jerarquia_v3.joblib')),
    os.path.abspath(os.path.join(os.getcwd(), 'data', 'processed', 'modelo_jerarquia_v3.joblib'))
]
modelo_path = next((p for p in candidatos_modelo if os.path.exists(p)), candidatos_modelo[1])
print(f"Ruta de modelo seleccionada: {modelo_path}")

In [ ]:
# 3. Lectura del Dataset de Test Curado
df_test = None

# Intentar lectura con PySpark en Databricks / Cluster
try:
    # Obtener o inicializar sesión de Spark si no está creada
    if 'spark' not in globals():
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName("MarchaBlancaClusterizacion").getOrCreate()

    config_key = f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net"
    spark.conf.set(config_key, storage_account_key)
    print(f"Intentando lectura desde Azure Blob Storage: {input_path_gold}")
    df_spark = spark.read.parquet(input_path_gold)
    n_spark = df_spark.count()
    print(f"Lectura exitosa desde Azure. Total registros en Spark: {n_spark:,}")
    df_test = df_spark.toPandas()
except Exception as e_spark:
    print(f"[Aviso] No se pudo leer desde Spark/WASBS: {e_spark}")
    if os.path.exists(local_fallback_path):
        print(f"Activando dataset local de contingencia: {local_fallback_path}")
        df_test = pd.read_parquet(local_fallback_path)
    else:
        raise FileNotFoundError(
            f"No se encontró el dataset en Azure Blob Storage ni en la ruta local: {local_fallback_path}. "
            "Asegúrate de que la ruta 'Clustering de Localidades test/' exista en el storage account."
        )

print(f"\nRegistros listos para evaluación de marcha blanca: {len(df_test):,}")
print(f"Columnas detectadas: {list(df_test.columns[:10])}...")

In [ ]:
# 4. Carga Directa y Protegida del Modelo Jerárquico (v3.0)
import time
import shutil

t0 = time.time()
print('=== VERIFICANDO ARTEFACTO DEL MODELO ===')

# 1. Lista de posibles ubicaciones en Databricks y local
candidatos = [
    modelo_path,
    '/tmp/modelo_jerarquia_v3.joblib',
    '../data/processed/modelo_jerarquia_v3.joblib',
    'data/processed/modelo_jerarquia_v3.joblib',
    '/dbfs/FileStore/models/modelo_jerarquia_v3.joblib',
    os.path.join(os.getcwd(), '..', 'data', 'processed', 'modelo_jerarquia_v3.joblib'),
    os.path.join(os.getcwd(), 'data', 'processed', 'modelo_jerarquia_v3.joblib')
]

ruta_encontrada = None
for c in candidatos:
    try:
        if c and os.path.isfile(c) and os.path.getsize(c) > 0:
            ruta_encontrada = os.path.abspath(c)
            break
    except Exception:
        continue

if not ruta_encontrada:
    print(' [ERROR] No se encontró el archivo modelo_jerarquia_v3.joblib en el cluster.')
    print('Causas posibles en Databricks:')
    print('  1. Databricks Repos no lo sincronizó (haz git commit y git pull en Databricks).')
    print('  2. O sube el archivo (170 KB) a DBFS FileStore o /tmp/ en el cluster.')
    print(f'Rutas inspeccionadas sin éxito: {candidatos[:4]}')
    raise FileNotFoundError('modelo_jerarquia_v3.joblib no encontrado en el sistema de archivos.')

print(f'Archivo localizado en: {ruta_encontrada} ({os.path.getsize(ruta_encontrada)/1024:.1f} KB)')

# 2. En Databricks, copiar al disco local efímero /tmp/ evita bloqueos de Workspace Files/FUSE
local_tmp = '/tmp/modelo_jerarquia_v3_fast.joblib'
if ruta_encontrada != local_tmp:
    try:
        shutil.copyfile(ruta_encontrada, local_tmp)
        ruta_carga = local_tmp
    except Exception:
        ruta_carga = ruta_encontrada
else:
    ruta_carga = ruta_encontrada

# 3. Cargar binario
print(f'Cargando payload con joblib desde: {ruta_carga}...')
with open(ruta_carga, 'rb') as f:
    payload_hier = joblib.load(f)

t_carga = time.time() - t0
print(f' ¡Modelo jerárquico cargado exitosamente en {t_carga:.2f} segundos!')
print(f'Versión: {payload_hier.get("version", "3.0")}')
print(f'Micro-clusters activos ({len(payload_hier.get("categorias_vigentes", []))}): {payload_hier.get("categorias_vigentes", [])[:6]}...')

In [ ]:
# 5. Ejecución de Inferencia Jerárquica (Nivel 1 Macro + Nivel 2 Micro)
print("Iniciando inferencia jerárquica defensiva sobre dataset de prueba...")
df_pred = predecir_microclusters(df_test, payload_hier)

# Incorporar predicciones al dataset
df_final_test = df_test.copy()
for col in ["micro_cluster_id", "label_auto", "arquetipo_demanda", "score_confianza", "es_frontera"]:
    df_final_test[col] = df_pred[col]

print("Inferencia completada exitosamente.")
display(df_final_test[['logical_seat_category', 'arquetipo_demanda', 'micro_cluster_id', 'label_auto', 'score_confianza', 'es_frontera']].head(10))

## 6. Diagnóstico y Distribución de Asignaciones en Marcha Blanca

In [ ]:
# Importar librerías de gráficos bajo demanda
import matplotlib.pyplot as plt
import seaborn as sns

# Distribución de localidades por Micro-Cluster y Macro-Arquetipo
resumen_mc = (
    df_final_test.groupby(['arquetipo_demanda', 'micro_cluster_id', 'label_auto'])
    .size()
    .reset_index(name='n_localidades')
)
resumen_mc['% del Test'] = (resumen_mc['n_localidades'] / len(df_final_test) * 100).round(2)
resumen_mc = resumen_mc.sort_values(by='n_localidades', ascending=False)

print('=== DISTRIBUCIÓN POR MICRO-CLUSTER EN TEST ===')
display(resumen_mc)

# Gráfico de distribución de micro-clusters en el test
plt.figure(figsize=(14, 5))
sns.barplot(
    data=resumen_mc[resumen_mc['micro_cluster_id'] != 'AU-0'],
    x='micro_cluster_id',
    y='n_localidades',
    hue='arquetipo_demanda',
    dodge=False,
    palette='tab10'
)
plt.title('Distribución de Nuevas Localidades por Micro-Cluster (Eventos Multi-Zona Test)', fontsize=12, fontweight='bold')
plt.xlabel('Micro-Cluster ID')
plt.ylabel('Número de Localidades')
plt.xticks(rotation=45)
plt.legend(title='Arquetipo Macro', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 7. Auditoría de Calidad: Drift Estadístico (PSI) e Incertidumbre

In [ ]:
# Evaluación formal de Population Stability Index (PSI)
res_drift = evaluar_drift_microclusters(df_final_test, payload_hier, umbral_psi=0.10)

print('=== AUDITORÍA DE DRIFT POBLACIONAL (PSI) ===')
print(f"PSI Global: {res_drift['psi_global']:.4f}")
print(f"Estado de Estabilidad: {res_drift['estado'].upper()}")
print(f"Umbral de Alerta: {res_drift['umbral_evaluado']}")

if res_drift.get('alertas'):
    print('\n[ALERTAS DE DESVIACIÓN]:')
    for al in res_drift['alertas']:
        print(f" - {al}")
else:
    print('\n[OK] La distribución de micro-clusters se mantiene alineada con la línea base histórica.')

# Tabla detallada de drift por micro-cluster
df_drift_det = pd.DataFrame(res_drift['distribucion_microclusters']).T
display(df_drift_det)

In [ ]:
# Análisis de Incertidumbre y Casos de Frontera
tasa_frontera = df_final_test['es_frontera'].mean() * 100
confianza_media = df_final_test['score_confianza'].mean()

print(f"Tasa de localidades en frontera de decisión: {tasa_frontera:.2f}% (Esperado < 20%)")
print(f"Score de confianza geométrico promedio: {confianza_media:.4f}")

print('\n=== TOP 10 CASOS CON MAYOR INCERTIDUMBRE (SCORE MÍNIMO) ===')
cols_rev = ['logical_seat_category', 'arquetipo_demanda', 'micro_cluster_id', 'label_auto', 'score_confianza', 'es_frontera']
cols_pres = [c for c in cols_rev if c in df_final_test.columns]
display(df_final_test[cols_pres].sort_values(by='score_confianza').head(10))

## 8. Persistencia de Predicciones en Gold Layer para Modelos Downstream

In [ ]:
# Guardar resultados enriquecidos en Azure Blob Storage (WASBS)
try:
    print(f"Escribiendo resultados de marcha blanca en Spark: {output_path_marcha_blanca}")
    df_spark_out = spark.createDataFrame(df_final_test)
    df_spark_out.write.mode('overwrite').parquet(output_path_marcha_blanca)
    print('Exportación a Azure Blob Storage completada exitosamente.')
except Exception as e:
    print(f"[Aviso] Spark no disponible ({e}). Guardando localmente en data/processed/")
    out_loc = '../data/processed/marcha_blanca_predicciones.parquet' if os.path.exists('../data/processed') else 'data/processed/marcha_blanca_predicciones.parquet'
    df_final_test.to_parquet(out_loc, index=False)
    print(f"Predicciones guardadas localmente en: {out_loc}")